# Setup

### Block 01 · Learning goal and AI prompt

**Learning objective:** Separate environment setup from analysis.

**Check after execution:** Did installation finish successfully?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Install geopandas, matplotlib, requests, folium and mapclassify in this notebook using a notebook-aware pip command. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
%pip install -q geopandas matplotlib requests folium mapclassify

## Exercise
1. Load the ISTAT **2026** province and municipality layers. Select Trento and plot it.
2. Find the largest and smallest municipalities by polygon area. Export their centroids and representative points in EPSG:4326.
3. Compare a planar centroid distance with its interpretation; it is not a route distance.
4. Reproduce and repair a missing-CRS error on the labelled training copy at the end.

# Solutions

## learning objectives
* repeat the concepts on the previous lesson
* introduce geopackage
* centroid vs representative point

---

### Import of the packages

### Block 02 · Learning goal and AI prompt

**Learning objective:** Recognise the roles of geospatial, tabular and plotting libraries.

**Check after execution:** Which library adds spatial methods to a table?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Import GeoPandas as gpd, pandas as pd, matplotlib.pyplot as plt, and Path from pathlib. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## ISTAT 2026: original source and a multilayer GeoPackage
Download the official generalised Shapefiles and build the GeoPackage locally. Each archive directory is a different administrative level.

https://www.istat.it/notizia/confini-delle-unita-amministrative-a-fini-statistici-al-1-gennaio-2018-2/

### Block 03 · Learning goal and AI prompt

**Learning objective:** Build a reusable multilayer dataset from original sources.

**Check after execution:** Which layers and CRS are stored in the file?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Download and extract the official ISTAT 2026 generalised ZIP, define the same source variables and read_istat helper as notebook 01, and create a local GeoPackage named istat_administrative_units_generalized_2026.gpkg in DATA. Map RipGeo to macroregions, Reg to regions, ProvCM to provinces and Com to municipalities. Reuse an existing complete file. Store its path as geopackage_istat_file and url. Use the official 2026 ZIP URL, not an invented remote GeoPackage link. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from zipfile import ZipFile

YEAR = 2026
ISTAT_URL = ("https://www.istat.it/storage/cartografia/"
             "confini_amministrativi/generalizzati/2026/Limiti01012026_g.zip")
DATA = Path("data_2026").resolve()
DATA.mkdir(exist_ok=True)
archive = DATA / "Limiti01012026_g.zip"
if not archive.exists():
    response = requests.get(ISTAT_URL, timeout=120)
    response.raise_for_status()
    archive.write_bytes(response.content)
source_dir = DATA / "istat"
if not list(source_dir.rglob("*.shp")):
    with ZipFile(archive) as z:
        z.extractall(source_dir)

def read_istat(prefix):
    candidates = list(source_dir.rglob(prefix + "*.shp"))
    if len(candidates) != 1:
        raise ValueError(f"Expected one {prefix} layer, found {candidates}")
    return gpd.read_file(candidates[0])

# Build the multilayer GeoPackage from the original 2026 Shapefiles.
geopackage_istat_file = str(DATA / "istat_administrative_units_generalized_2026.gpkg")
if not Path(geopackage_istat_file).exists():
    for layer, prefix in {"macroregions": "RipGeo", "regions": "Reg",
                          "provinces": "ProvCM", "municipalities": "Com"}.items():
        read_istat(prefix).to_file(geopackage_istat_file, layer=layer, driver="GPKG")
url = geopackage_istat_file


### Block 04 · Learning goal and AI prompt

**Learning objective:** Inspect provenance and layer structure together.

**Check after execution:** Can someone identify both the source and the local layers?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Print ISTAT_URL and the GeoPackage layer listing from geopackage_istat_file. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
print(ISTAT_URL)
print(gpd.list_layers(geopackage_istat_file))

## GeoPackage
One SQLite-based file can store multiple vector layers, CRS information and attributes. We have created `macroregions`, `regions`, `provinces` and `municipalities` from the official 2026 archive.

Reference: https://www.geopackage.org/

Use `gpd.list_layers(geopackage_istat_file)` to inspect the contents.

Geopandas can manage geopackage by using [pyogrio](https://pyogrio.readthedocs.io/en/latest/)

### Block 05 · Learning goal and AI prompt

**Learning objective:** Distinguish a storage format from its software driver.

**Check after execution:** What does a driver describe?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Import list_drivers from pyogrio. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
from pyogrio import list_drivers


### Block 06 · Learning goal and AI prompt

**Learning objective:** Check supported read and write capabilities.

**Check after execution:** Does GPKG support both reading and writing?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the available pyogrio drivers with list_drivers. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
list_drivers()


```
'GPKG': 'rw',
```
**rw** => **r**ead **w**rite

geopandas can:
* **r**ead *geopackage* files
* **w**rite data to a *geopackage* file

geopackage can store more layers, so we have to investigate the contents

### Block 07 · Learning goal and AI prompt

**Learning objective:** Inspect containers before choosing their contents.

**Check after execution:** Why does a multilayer file need layer discovery?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Import list_layers from pyogrio. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
from pyogrio import list_layers


### Block 08 · Learning goal and AI prompt

**Learning objective:** Identify the actual layers in a GeoPackage.

**Check after execution:** Which layer contains provinces?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List layers in geopackage_istat_file using list_layers. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
list_layers(geopackage_istat_file)

### Block 09 · Learning goal and AI prompt

**Learning objective:** Read the intended administrative level.

**Check after execution:** Does each row represent a province rather than a municipality?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Load the provinces layer of geopackage_istat_file into provinces. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces = gpd.read_file(geopackage_istat_file, layer="provinces")

*TIP*: you can also download the file directly from geopandas with this function

### Block 10 · Learning goal and AI prompt

**Learning objective:** Understand that read_file accepts a path variable.

**Check after execution:** Is url actually a remote download in this notebook?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the provinces layer from url into provinces. Here url is a local GeoPackage path established earlier. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces = gpd.read_file(url,layer="provinces")

### Block 11 · Learning goal and AI prompt

**Learning objective:** Inspect provincial observations before filtering.

**Check after execution:** Which fields distinguish provinces and metropolitan cities?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the first three rows of provinces. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces.head(3)

### Block 12 · Learning goal and AI prompt

**Learning objective:** Read a schema before selecting a field.

**Check after execution:** Which field contains province names?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display provinces.columns. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces.columns

**COD_RIP**<br/>
*codice ripartizione*<br/>
numeric code of the macroregion of belonging

**COD_REG**<br/>
*codice regione*<br/>
numeric code of the region of belonging

**COD_PROV**<br/>
*codice provincia*<br/>
numeric code of the region of belonging

**COD_CM**<br/>
*codice comune*<br/>
Istat code of the metropolitan city (three characters in
reference to all official statistics are
numeric format) obtained by adding the value 200 to
adopted the statistical codes of the cities
corresponding code of the province.

**COD_UTS**<br/>
*codice unità territoriali sovracomunali*<br/>
Numeric code that uniquely identifies the Units
territorial supra-municipal on the national territory.

**DEN_PROV**<br/>
*denominazione provincia*<br/>
name of the province

**DEN_CM**<br/>
*denominazione città metropolitana*<br/>
name of the metropolitan city

**DEN_UTS**<br/>
*denominazione unità territoriale sovracomunale*<br/>
Denomination of the supra-municipal territorial units.

**SIGLA**<br/>
*sigla*<br/>
abbreviation

**TIPO_UTS**<br/>
*tipologia unità territoriale sovracomunale*<br/>
kind of supra-municipal territorial units.



### filter it for an italian provice at your choice (eg. Trento)

obtain the list of all the provincies

### Block 13 · Learning goal and AI prompt

**Learning objective:** Inspect available province names.

**Check after execution:** What is the exact value for Trento?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List unique DEN_PROV values in provinces. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces['DEN_PROV'].unique()

obtain the list of all the metropolitan ciies

### Block 14 · Learning goal and AI prompt

**Learning objective:** Distinguish provincial and metropolitan-city attributes.

**Check after execution:** Why are some names or values absent?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List unique DEN_CM values in provinces. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces.DEN_CM.unique()

choose the province of Trento

### Block 15 · Learning goal and AI prompt

**Learning objective:** Filter the province by an explicit attribute.

**Check after execution:** Did the filter retain the expected administrative unit?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create province_of_trento from provinces where DEN_PROV equals Trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
province_of_trento = provinces[provinces['DEN_PROV']=='Trento']

### Block 16 · Learning goal and AI prompt

**Learning objective:** Inspect the selected province and retained attributes.

**Check after execution:** Which identifier should be retained for integration?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display province_of_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
province_of_trento

### Block 17 · Learning goal and AI prompt

**Learning objective:** Check the selected layer’s measurement units.

**Check after execution:** Are the coordinate units suitable for planar measurements?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display province_of_trento.crs. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
province_of_trento.crs

### Block 18 · Learning goal and AI prompt

**Learning objective:** Check that filtering preserves geospatial structure.

**Check after execution:** Is it still a GeoDataFrame?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the type of province_of_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
type(province_of_trento)

### plot it

### Block 19 · Learning goal and AI prompt

**Learning objective:** Verify the selected province on a map.

**Check after execution:** Does the geometry match the territory selected?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot province_of_trento in a 10 by 10 inch figure and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
province_of_trento.plot(figsize=(10,10))
plt.show()

## 2. load the shapefile of ISTAT with the informations of the muncipalities
### identify the municipalities of the province selected with the biggest and smallest area

this means we need to use another layer / dataset

### Block 20 · Learning goal and AI prompt

**Learning objective:** Choose the municipality layer independently of the province layer.

**Check after execution:** Does this layer have a different observation unit?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the municipalities layer from geopackage_istat_file into municipalities. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities = gpd.read_file(geopackage_istat_file, layer="municipalities")

### Block 21 · Learning goal and AI prompt

**Learning objective:** Identify municipality and province keys in the schema.

**Check after execution:** Which fields identify the municipality and its province?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display municipalities.columns. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities.columns

**COD_RIP**<br/>
*codice ripartizione*<br/>
numeric code of the macroregion of belonging

**COD_REG**<br/>
*codice regione*<br/>
numeric code of the region of belonging

**COD_PROV**<br/>
*codice provincia*<br/>
numeric code of the region of belonging

**COD_CM**<br/>
*codice comune*<br/>
unique numeric identification code of the municipality within the province of belonging

**COD_UTS**<br/>
*codice unità territoriali sovracomunali*<br/>
Numeric code that uniquely identifies the Units
territorial supra-municipal on the national territory.

**PRO_COM**<br/>
*provincia comune**<br/>
Numeric code that uniquely identifies the Municipality
on the national territory. (= COD_PROV & COD_COM)

**PRO_COM_T**<br/>
*provincia comune territorio**<br/>
Alphanumeric code that uniquely identifies the
Municipality on the national territory.<br/>
Like PRO_COM but definied in 6 fixed characters.

**COMUNE**<br/>
*comune*<br/>
Name of the Municipality

**COMUNE_A**<br/>
*comune alternativa*<br/>
Name of the Municipality in a language other than Italian

**CC_UTS**<br/>
*comune capoluogo*<br/>
Provincial capital or metropolitan city<br/>
1 => True<br/>
2 => False



the **COD_PROV** of the Province of Trento is **22**

### Block 22 · Learning goal and AI prompt

**Learning objective:** Use a province code to select municipalities.

**Check after execution:** Does every selected record belong to Trento province?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create municipalities_province_trento by filtering municipalities where COD_PROV equals 22. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# filter the province
municipalities_province_trento= municipalities[municipalities.COD_PROV==22]

### Block 23 · Learning goal and AI prompt

**Learning objective:** Check the municipality subset visually.

**Check after execution:** Does the subset cover the province as expected?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot municipalities_province_trento with a 10 by 10 inch figure and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# plot it
municipalities_province_trento.plot(figsize=(10,10))
plt.show()

### identify the cities of the province selected with the biggest and smallest area

**CAUTION**:

we are using generalized boundaries !!!

finding the max area

### Block 24 · Learning goal and AI prompt

**Learning objective:** Find an extreme using computed geometry measurements.

**Check after execution:** What units does this value use?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the maximum polygon area of municipalities_province_trento and save it as max_area. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
max_area = municipalities_province_trento.geometry.area.max()

### Block 25 · Learning goal and AI prompt

**Learning objective:** Inspect the maximum area directly.

**Check after execution:** Does it agree with max_area?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the maximum geometry area in municipalities_province_trento without assigning a new variable. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities_province_trento.geometry.area.max()

### Block 26 · Learning goal and AI prompt

**Learning objective:** Check measurement assumptions before interpreting a ranking.

**Check after execution:** Would the same area calculation be valid in degrees?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Print municipalities_province_trento.crs. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# Keep the result reproducible: inspect computed areas, not old labels.
print(municipalities_province_trento.crs)

### Block 27 · Learning goal and AI prompt

**Learning objective:** Read a stored analytical result.

**Check after execution:** Is this the value of an area or the name of a municipality?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display max_area. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
max_area

finding the min area

### Block 28 · Learning goal and AI prompt

**Learning objective:** Find the minimum under the same measurement assumptions.

**Check after execution:** Is it comparable with max_area in the same units?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the minimum polygon area of municipalities_province_trento as min_area. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
min_area = municipalities_province_trento.geometry.area.min()

### Block 29 · Learning goal and AI prompt

**Learning objective:** Inspect the computed minimum.

**Check after execution:** Is it positive and expressed in the expected units?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display min_area. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
min_area

... you can obtain the same in another way (combination of the requests)

### Block 30 · Learning goal and AI prompt

**Learning objective:** Recognise equivalent filter-and-aggregation expressions.

**Check after execution:** Does it equal min_area?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Filter municipalities by COD_PROV 22 and calculate the minimum geometry area in one expression. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities[municipalities.COD_PROV==22].geometry.area.min()

### identify the municipality with the biggest area




### Block 31 · Learning goal and AI prompt

**Learning objective:** Recover the observation that produced an extreme.

**Check after execution:** Could several records tie for the maximum?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select every municipality in municipalities_province_trento whose geometry area equals max_area into maxarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento = municipalities_province_trento[municipalities_province_trento.geometry.area == max_area]

### Block 32 · Learning goal and AI prompt

**Learning objective:** Preserve stable identifiers in a local subset.

**Check after execution:** Are leading zeros and fixed-width codes preserved?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display PRO_COM_T for municipalities_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities_province_trento.PRO_COM_T

### Block 33 · Learning goal and AI prompt

**Learning objective:** Identify the largest municipality by its key.

**Check after execution:** How could this key join another table?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display PRO_COM_T for maxarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.PRO_COM_T

### Block 34 · Learning goal and AI prompt

**Learning objective:** Derive names from the selected data rather than old claims.

**Check after execution:** Does the 2026 result agree with the old teaching example?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display COMUNE for maxarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.COMUNE

The largest municipality is **computed below**, using the current 2026 geometries. Do not retain the old notebook’s hard-coded claim that it is Trento.

### Block 35 · Learning goal and AI prompt

**Learning objective:** Validate the largest-area selection visually.

**Check after execution:** Is the selected geometry consistent with its name and extent?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot maxarea_municipality_province_trento and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.plot(figsize=(10,10))
plt.show()

### Block 36 · Learning goal and AI prompt

**Learning objective:** Inspect the largest municipality interactively.

**Check after execution:** What does the interactive map add to the static plot?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create an interactive explore map of maxarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.explore()

### identify the municipality with the smallest area


### Block 37 · Learning goal and AI prompt

**Learning objective:** Recover the smallest-area observation.

**Check after execution:** Could a tie produce more than one record?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select municipalities whose geometry area equals min_area into minarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
minarea_municipality_province_trento = municipalities_province_trento[municipalities_province_trento.geometry.area == min_area]

### Block 38 · Learning goal and AI prompt

**Learning objective:** Read the name from the selected minimum.

**Check after execution:** Was the answer computed rather than hard-coded?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display COMUNE for minarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
minarea_municipality_province_trento.COMUNE

### Block 39 · Learning goal and AI prompt

**Learning objective:** Keep a key for the minimum-area municipality.

**Check after execution:** Does it remain comparable with other official municipality codes?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display PRO_COM_T for minarea_municipality_province_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
minarea_municipality_province_trento.PRO_COM_T

The smallest municipality is also determined from the data. These are areas of generalised statistical boundaries.

### Block 40 · Learning goal and AI prompt

**Learning objective:** Verify the smallest-area municipality on a map.

**Check after execution:** Can figure autoscaling hide the size difference from the largest municipality?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot minarea_municipality_province_trento and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
minarea_municipality_province_trento.plot(figsize=(10,10))
plt.show()

### extract all the centroids of the areas expressed in WGS84

### Block 41 · Learning goal and AI prompt

**Learning objective:** Observe the coordinates of a projected centroid after transformation.

**Check after execution:** Does this repeat of the calculation change its interpretation or position?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the centroid of maxarea_municipality_province_trento in its existing projected CRS, then transform the resulting point to EPSG:4326. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.geometry.centroid.to_crs(epsg=4326)

### Block 42 · Learning goal and AI prompt

**Learning objective:** Observe the coordinates of a projected centroid after transformation.

**Check after execution:** Does this repeat of the calculation change its interpretation or position?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the centroid of maxarea_municipality_province_trento in its existing projected CRS, then transform the resulting point to EPSG:4326. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.geometry.centroid.to_crs(epsg=4326)

### Block 43 · Learning goal and AI prompt

**Learning objective:** Observe the coordinates of a projected centroid after transformation.

**Check after execution:** Does this repeat of the calculation change its interpretation or position?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the centroid of maxarea_municipality_province_trento in its existing projected CRS, then transform the resulting point to EPSG:4326. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.geometry.centroid.to_crs(epsg=4326)

### Block 44 · Learning goal and AI prompt

**Learning objective:** Consult method documentation before choosing a geometric summary.

**Check after execution:** What guarantee does a representative point provide?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display Python help for the representative_point method on maxarea_municipality_province_trento.geometry. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
help(maxarea_municipality_province_trento.geometry.representative_point)

### extract a represenative point for the area of the smallest and bigger municipality in WGS84

### Block 45 · Learning goal and AI prompt

**Learning objective:** Construct an interior point for the correct selected municipality.

**Check after execution:** Did the code use the minimum-area subset rather than the maximum?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate representative points of minarea_municipality_province_trento in its source CRS as representative_point_minarea_municipality. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
representative_point_minarea_municipality = minarea_municipality_province_trento.geometry.representative_point()

### Block 46 · Learning goal and AI prompt

**Learning objective:** Apply the same operation to the maximum-area subset.

**Check after execution:** Are minimum and maximum variables based on different inputs?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate representative points of maxarea_municipality_province_trento in its source CRS as representative_point_maxarea_municipality. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
representative_point_maxarea_municipality = maxarea_municipality_province_trento.geometry.representative_point()

### Block 47 · Learning goal and AI prompt

**Learning objective:** Transform an already constructed representative point.

**Check after execution:** Were point construction and CRS transformation kept in the correct order?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform representative_point_minarea_municipality to EPSG:4326 and display it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
representative_point_minarea_municipality.to_crs(epsg=4326)

### Block 48 · Learning goal and AI prompt

**Learning objective:** Transform polygon coordinates without changing the observation unit.

**Check after execution:** Is the geometry still a polygon rather than a centroid?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display maxarea_municipality_province_trento reprojected to EPSG:4326. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.to_crs(epsg=4326)

### save the data in GeoJSON files

we can save each point in geojson

### Block 49 · Learning goal and AI prompt

**Learning objective:** Write a point layer in a web-compatible CRS.

**Check after execution:** Which attributes, if any, are retained by this GeoSeries export?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform representative_point_maxarea_municipality to EPSG:4326 and export point.geojson using the GeoJSON driver. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
representative_point_maxarea_municipality.to_crs(epsg=4326).to_file("point.geojson",driver="GeoJSON")

... but we need to create an only one file with all the data in a geojson file

### Block 50 · Learning goal and AI prompt

**Learning objective:** Combine comparable derived points.

**Check after execution:** Do the two inputs share a CRS?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Concatenate representative_point_minarea_municipality and representative_point_maxarea_municipality into points, resetting their index. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
points = pd.concat([representative_point_minarea_municipality, representative_point_maxarea_municipality], ignore_index=True)

### Block 51 · Learning goal and AI prompt

**Learning objective:** Inspect the combined point collection.

**Check after execution:** Does it contain the intended distinct observations?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display points. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
points

### Block 52 · Learning goal and AI prompt

**Learning objective:** Export the combined points.

**Check after execution:** Can you reopen the same number of point features?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform points to EPSG:4326 and save points.geojson using the GeoJSON driver. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
points.to_crs(epsg=4326).to_file("points.geojson",driver="GeoJSON")

and we can do the same with the polygons

### Block 53 · Learning goal and AI prompt

**Learning objective:** Combine polygon observations while retaining attributes.

**Check after execution:** How does this differ from dissolving the geometries?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Concatenate maxarea_municipality_province_trento and minarea_municipality_province_trento into polygons with a fresh index. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
polygons = pd.concat([maxarea_municipality_province_trento, minarea_municipality_province_trento], ignore_index=True)

### Block 54 · Learning goal and AI prompt

**Learning objective:** Export selected polygons without converting them to points.

**Check after execution:** Does the output geometry type remain polygonal?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform polygons to EPSG:4326 and write polygons.geojson. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
polygons.to_crs(epsg=4326).to_file("polygons.geojson",driver="GeoJSON")

### Block 55 · Learning goal and AI prompt

**Learning objective:** Inspect the merged polygon table.

**Check after execution:** Are municipality names and identifiers still available?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display polygons. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
polygons

Open https://web.geolibre.app/ and add the exported GeoJSON layers. Alternatively use https://umap.openstreetmap.fr/ . Inspect the points against their polygons.

### Projected straight-line distance between centroids
This is a planar distance in the source projected CRS, **not a geodesic or a route distance**.

### Block 56 · Learning goal and AI prompt

**Learning objective:** Choose an explicit destination for a distance calculation.

**Check after execution:** Is this a geometric point rather than a settlement location?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the centroid of the smallest-area municipality in its projected CRS and store the first geometry as to_point. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
to_point = minarea_municipality_province_trento.geometry.centroid.values[0]

### Block 57 · Learning goal and AI prompt

**Learning objective:** Interpret distance as a planar geometric measurement.

**Check after execution:** Is this a straight-line projected distance, a geodesic or a route?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate distances from centroids of maxarea_municipality_province_trento to to_point. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
maxarea_municipality_province_trento.centroid.distance(to_point)

### Block 58 · Learning goal and AI prompt

**Learning objective:** Store a scalar result from a vectorised distance calculation.

**Check after execution:** What assumptions underlie selecting the first result?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the projected centroid distance from the maximum-area municipality to to_point and store its first value as distance. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
distance = maxarea_municipality_province_trento.geometry.centroid.distance(to_point).values[0]

### Block 59 · Learning goal and AI prompt

**Learning objective:** Read the distance in source units.

**Check after execution:** Are the units metres?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display distance. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
distance

the distance is in meters due the CRS used on the dataset

### Block 60 · Learning goal and AI prompt

**Learning objective:** Prepare units and labels for communicating a result.

**Check after execution:** Do the labels match the two geometries used?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Set to_city and from_city from the minimum- and maximum-area municipality names and convert distance to kilometres as distance_km. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
to_city = minarea_municipality_province_trento.COMUNE.values[0]
from_city = maxarea_municipality_province_trento.COMUNE.values[0]
distance_km = distance/1000

### Block 61 · Learning goal and AI prompt

**Learning objective:** Communicate a computed distance with units.

**Check after execution:** Could the sentence be mistaken for a driving distance?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Print from_city, to_city and distance_km in one sentence rounded to one decimal place. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
print("the distance from %s to %s is %.1f km" % (from_city, to_city, round(distance_km,1)))

### Why a representative point?

Where is the centroid of Liguria?


### Block 62 · Learning goal and AI prompt

**Learning objective:** Load a different administrative level for a counterexample.

**Check after execution:** Why is a region useful for demonstrating centroid behaviour?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the regions layer from url into regions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions = gpd.read_file(url,layer="regions")

### Block 63 · Learning goal and AI prompt

**Learning objective:** Inspect exact region labels before selecting.

**Check after execution:** Which exact value selects Liguria?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List unique DEN_REG values in regions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions.DEN_REG.unique()

### Block 64 · Learning goal and AI prompt

**Learning objective:** Read the geometry that makes centroid interpretation difficult.

**Check after execution:** Could a geometric centroid fall outside this shape?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select Liguria by DEN_REG in regions, plot it and show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions[regions.DEN_REG=='Liguria'].plot(figsize=(10,10))
plt.show()

### Block 65 · Learning goal and AI prompt

**Learning objective:** Export the polygon needed to check derived points.

**Check after execution:** Can the polygon be compared with both point exports?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select Liguria, transform its polygons to EPSG:4326 and save liguria.geojson. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions[regions.DEN_REG=='Liguria'].to_crs(epsg=4326).to_file("liguria.geojson",driver='GeoJSON')

### Block 66 · Learning goal and AI prompt

**Learning objective:** Construct the centroid before changing CRS.

**Check after execution:** Does this guarantee an interior point?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select Liguria, calculate its centroid in the source projected CRS, transform that point to EPSG:4326 and write liguria_centroid.geojson. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions[regions.DEN_REG=='Liguria'].centroid.to_crs(epsg=4326).to_file("liguria_centroid.geojson",driver='GeoJSON')

### Block 67 · Learning goal and AI prompt

**Learning objective:** Construct and export a point with an interior guarantee.

**Check after execution:** How does this point differ from the centroid?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select Liguria, calculate representative points in the source CRS, transform them to EPSG:4326 and write liguria_representative_point.geojson. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions[regions.DEN_REG=='Liguria'].representative_point().to_crs(epsg=4326).to_file("liguria_representative_point.geojson",driver='GeoJSON')

Compare the centroid and representative point in GeoLibre. The following historical illustration is retained from the original teaching material; regenerate the analysis from 2026 data.

![](https://raw.githubusercontent.com/napo/geospatial_course_unitn/master/images/umap_liguria.jpg)


## Missing CRS: a controlled exercise
The old lakes download is unavailable. Reproduce the missing-CRS problem on a **training copy** of ISTAT municipalities. The official source is not altered. We know its CRS from the source `.prj`.

### Block 68 · Learning goal and AI prompt

**Learning objective:** Recognise how missing metadata prevents a valid transformation.

**Check after execution:** Was the official source kept unchanged?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Save the known CRS as source_crs, copy municipalities_province_trento into training and deliberately remove the CRS only on that training copy. Print training.crs. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
source_crs = municipalities_province_trento.crs
training = municipalities_province_trento.copy()
training.geometry = gpd.GeoSeries(training.geometry.to_numpy(), index=training.index)
training = training.set_crs(None, allow_override=True)
print(training.crs)

### Block 69 · Learning goal and AI prompt

**Learning objective:** Interpret an expected CRS error instead of hiding it.

**Check after execution:** Does an error identify a missing declaration or wrong coordinates?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Attempt to transform training to EPSG:4326 in a try/except block and print the ValueError. Do not repair it in this cell. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
try:
    training.to_crs(4326)
except ValueError as error:
    print(type(error).__name__, str(error))

Assign the **known source CRS** first. Only then transform coordinates. Never infer a CRS from a plausible-looking map.

### Block 70 · Learning goal and AI prompt

**Learning objective:** Distinguish assigning known metadata from transforming coordinates.

**Check after execution:** What evidence justifies the assigned source CRS?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Assign source_crs to training, then transform it to EPSG:4326 as training_wgs84 and export training_municipalities.geojson. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
training = training.set_crs(source_crs)
training_wgs84 = training.to_crs(4326)
training_wgs84.to_file("training_municipalities.geojson", driver="GeoJSON")

## Next: pharmacies
Use the original national CSV, not a prepared Trentino subset. Notebook 03 filters validity dates, handles decimal commas and missing coordinates, and compares declared and spatial municipalities.

https://www.dati.salute.gov.it/it/dataset/farmacie/